# ISOM 839 · Session 12 — Reinforcement Learning & the AI Frontier of Prescriptive Analytics

**Prescriptive Analytics: Modeling & Optimization · Suffolk University · Prof. Hasan Arslan**

Last week we predicted, then decided. Tonight the two stages collapse into one: an **agent** that learns good decisions directly from experience and reward, with no forecast anywhere. Then the newest layer of the stack — language models that draft your optimization model — and the one skill that matters more than ever: **verifying what the copilot wrote.**

In [ ]:
%pip install -q gymnasium gurobipy scipy
import numpy as np, gymnasium as gym
import matplotlib.pyplot as plt
import gurobipy as gp
from gurobipy import GRB
np.random.seed(839)

---
## Part 1 — A tiny inventory world (tonight's lab)

**State:** units on hand, 0–20. **Action:** order 0–10 units, delivered overnight. **Demand:** Poisson with mean 5, capped by what is on the shelf. **Reward per day:** $10 per unit sold − $6 per unit ordered − $1 per unit held overnight − $4 per unit of unmet demand. An episode is 30 days. The agent sees none of these numbers — only states, actions and rewards.

In [ ]:
class TinyInventoryEnv(gym.Env):
    MAX_INV, MAX_ORDER, HORIZON = 20, 10, 30
    PRICE, COST, HOLD, PENALTY, MEAN_DEMAND = 10, 6, 1, 4, 5
    def __init__(self):
        self.observation_space = gym.spaces.Discrete(self.MAX_INV + 1)
        self.action_space = gym.spaces.Discrete(self.MAX_ORDER + 1)
        self.rng = np.random.default_rng(839)
    def reset(self, seed=None, options=None):
        self.inv, self.day = 5, 0
        return self.inv, {}
    def step(self, a):
        stock = min(self.inv + a, self.MAX_INV)           # shelf cap: extra units are turned away
        d = self.rng.poisson(self.MEAN_DEMAND)
        sold = min(stock, d); left = stock - sold
        r = self.PRICE * sold - self.COST * (stock - self.inv) - self.HOLD * left - self.PENALTY * (d - sold)
        self.inv, self.day = left, self.day + 1
        return self.inv, r, self.day >= self.HORIZON, False, {}

env = TinyInventoryEnv()
s, _ = env.reset(); s2, r, done, _, _ = env.step(3)
print(f'start with {s} units, order 3 -> next morning {s2} units, reward ${r}')

---
## Part 2 — Q-learning: a table of "how good is this action here?"

Keep a table Q[state, action]. Act (mostly greedily, sometimes at random), observe the reward and the next state, and nudge the entry toward *reward + γ × the best you can do next*. Repeat for 5,000 episodes. That one-line update is the entire algorithm; everything else is bookkeeping.

In [ ]:
def q_learn(env, episodes=5000, alpha=0.1, gamma=0.95, eps=1.0, eps_min=0.05, eps_decay=0.999, seed=839):
    rng = np.random.default_rng(seed)
    Q = np.zeros((env.MAX_INV + 1, env.MAX_ORDER + 1))
    curve = []
    for ep in range(episodes):
        s, _ = env.reset(); done = False; total = 0.0
        while not done:
            a = rng.integers(env.MAX_ORDER + 1) if rng.random() < eps else int(np.argmax(Q[s]))
            s2, r, done, _, _ = env.step(a)
            Q[s, a] += alpha * (r + gamma * np.max(Q[s2]) - Q[s, a])     # the Q-learning update
            s, total = s2, total + r
        eps = max(eps_min, eps * eps_decay)
        curve.append(total)
    return Q, np.array(curve)

Q, curve = q_learn(env)
policy = np.argmax(Q, axis=1)
print('learned order quantity at stock 0..14:', list(map(int, policy[:15])))
print('(stock above ~15 is almost never visited, so those rows of Q are still guesses)')

In [ ]:
plt.figure(figsize=(7, 3.4))
window = 100
plt.plot(np.convolve(curve, np.ones(window) / window, mode='valid'))
plt.xlabel('episode'); plt.ylabel(f'reward per 30-day episode (avg of {window})'); plt.title('The climb is the learning'); plt.grid(alpha=.3); plt.show()

**Read the policy.** The agent orders a lot when the shelf is empty and nothing once it holds a week's demand — a base-stock rule nobody taught it, learned from reward alone. It is noisier than Session 10's value iteration because the agent *estimated* the world instead of being handed it. How close did it get? Solve the same world exactly with value iteration (you know the Poisson demand; the agent did not) and compare:

In [ ]:
from scipy.stats import poisson
pmf = poisson.pmf(np.arange(0, 21), env.MEAN_DEMAND); pmf[-1] += 1 - pmf.sum()   # truncate the tail
V = np.zeros(21)
for _ in range(2000):
    V_new = np.full(21, -np.inf); exact = np.zeros(21, int)
    for s in range(21):
        for a in range(11):
            stock = min(s + a, 20); q = 0.0
            for d, p in enumerate(pmf):
                sold = min(stock, d); left = stock - sold
                q += p * (env.PRICE * sold - env.COST * (stock - s) - env.HOLD * left - env.PENALTY * (d - sold) + 0.95 * V[left])
            if q > V_new[s]: V_new[s], exact[s] = q, a
    if np.max(np.abs(V_new - V)) < 1e-6: break
    V = V_new
print('exact (value iteration):', list(map(int, exact[:15])))
print('learned (Q-learning):   ', list(map(int, policy[:15])))

The agent is within a unit or two of the exact policy on the states it visits often, and wrong on the ones it rarely sees. That is what learning from experience looks like: **good where the data is, guessing where it is not.** Deployed RL systems spend most of their engineering on exactly that gap.

**Step 4 — stress-test the agent.** Change the world and retrain. No reformulation, no new forecast — just new experience.

In [ ]:
class HighHoldingEnv(TinyInventoryEnv):
    HOLD = 4                                            # holding cost quadruples
Q2, _ = q_learn(HighHoldingEnv())
print('holding $1:', list(map(int, np.argmax(Q, axis=1)[:8])), '...  (order quantity at stock 0..7)')
print('holding $4:', list(map(int, np.argmax(Q2, axis=1)[:8])), '...')

### 💬 Discussion
For which decisions in *your* organization is there enough repetition and feedback for this loop to work — and for which would 5,000 episodes of learning by trial cost real money? That boundary is where RL is deployed (Alibaba, Amazon, DoorDash: millions of SKUs, daily feedback) and where it is not (a plant location you decide once).

---
## Part 3 — The LLM copilot: it writes, you verify

The newest layer: a language model reads a business problem in English and drafts the optimization model. Your job shifts from typing constraints to **judging them**. Below is a model an LLM drafted for the following prompt. It runs. It returns a number. It is wrong — twice.

> *"A bakery makes bread ($30 profit each) and cake ($50). Bread needs 1 hour of oven time and 2 lb of flour; cake needs 2 hours and 1 lb. There are 40 oven-hours and 50 lb of flour per day. Cake demand is at most 15 a day. Maximize profit."*

In [ ]:
# --- Drafted by an LLM. Do not fix it yet — find the bugs first. ---
m = gp.Model('bakery_llm'); m.Params.OutputFlag = 0
bread = m.addVar(name='bread'); cake = m.addVar(name='cake')
m.setObjective(30 * bread + 50 * cake, GRB.MAXIMIZE)
m.addConstr(2 * bread + 1 * cake <= 40, 'oven')
m.addConstr(1 * bread + 2 * cake <= 50, 'flour')
m.addConstr(bread <= 15, 'cake_demand')
m.optimize()
print(f'LLM model says: profit ${m.ObjVal:,.0f}  bread {bread.X:.0f}  cake {cake.X:.0f}')

**The verification checklist** — run it on every model a copilot hands you, including your final project's:

1. **Units check:** read each constraint aloud with units. "2 × bread + 1 × cake ≤ 40 oven-hours" — bread needs *1* hour, not 2. The coefficients are swapped between oven and flour.
2. **Name check:** the constraint called `cake_demand` bounds *bread*.
3. **Sanity solve:** set every constraint aside but one and confirm the answer moves the way you expect.
4. **Shadow-price check:** are the binding constraints the ones you would expect to bind? If flour is slack while the bakery is turning away cake orders, something is wrong.
5. **Defend each line:** can you say, for every constraint, which sentence of the prompt it came from? If a constraint has no sentence, delete it; if a sentence has no constraint, add it.

Now fix it:

In [ ]:
# TODO: correct the two bugs, re-solve, and compare the two optimal plans and profits
# (Expected after the fix: oven 1*bread + 2*cake <= 40, flour 2*bread + 1*cake <= 50, cake <= 15.)

### Try the real thing
Paste the bakery prompt into ChatGPT, Claude or Gurobi AI Modeling and run the draft here. Then paste in *your final project's* decision description and read the draft with the five-point checklist. The copilot is fast; you are accountable. On presentation night, every team member answers two questions about their own constraints — without the AI.

---
## The modern prescriptive stack, in one picture

| Weeks | You learned to | The decision looks like |
|---|---|---|
| 1–3 | Formulate and solve LPs; read shadow prices | *How much* of each thing |
| 4–5 | Network flows | *Where* things move |
| 6–7 | Integer and constraint programming | *Whether* — open, assign, schedule |
| 8 | Convex optimization | Risk and curves |
| 9–10 | Markov chains and MDPs | *Over time*, under uncertainty |
| 11 | Predict-then-optimize | With a forecast in the loop |
| 12 | Reinforcement learning, LLM copilots | Learned from experience; drafted by AI, verified by you |

It is not twelve topics. It is one toolkit, and you own all of it. **Final project kit:** [isom-839.vercel.app/project](https://isom-839.vercel.app/project) — the starter notebook, the memo template, and the model-defense question bank. See you on presentation night. 🏆